In [ ]:
# Ch03: 트랜스포머와 LLM — GPT에게 OPF 풀게 하기
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                          'pandapower'])
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

try:
    sys.path.insert(0, '..')
    from utils.style import set_style, COLORS
    set_style()
except:
    pass

## 1. OPF 솔버로 기준해 구하기

pandapower로 IEEE 14-bus OPF를 실행하여 정답(기준해)을 구합니다.

In [ ]:
import pandapower as pp
import pandapower.networks as pn

net = pn.case14()
pp.runopp(net, verbose=False)

gen_results = net.res_gen[['p_mw', 'q_mvar']].round(2)
bus_voltages = net.res_bus['vm_pu'].round(4)
total_cost = net.res_cost

print("=== OPF 솔버 결과 ===")
print(f"슬랙 버스(ext_grid) 출력:\n{net.res_ext_grid[['p_mw', 'q_mvar']].round(2)}")
print(f"\n발전기 출력:\n{gen_results}")
print(f"\n모선 전압 (p.u.):\n{bus_voltages.to_string()}")
print(f"\n총 발전 비용: {total_cost:.2f} $/h")

## 2. LLM 프롬프트 전략 비교

동일한 OPF 문제를 세 가지 방식으로 LLM에게 제시합니다: (A) 직접 수치, (B) CoT, (C) 코드 생성

In [ ]:
# TODO: 세 가지 프롬프트를 작성하세요

# 방법 A: 직접 수치 답변 요청
prompt_a = None  # <-- 프롬프트를 작성하세요

# 방법 B: 풀이 과정 설명 요청 (Chain-of-Thought)
prompt_b = None  # <-- 프롬프트를 작성하세요

# 방법 C: Python 코드 생성 요청
prompt_c = None  # <-- 프롬프트를 작성하세요

## 3. LLM 응답 시뮬레이션 (Pre-recorded)

API 키 없이도 실습할 수 있도록 미리 기록된 응답을 사용합니다.

In [ ]:
# Pre-recorded LLM 응답 (API 키 없이 실습 가능)
llm_response_a = """
방법 A 결과 (LLM 직접 답변 — 부정확):
- Gen 1: ~200 MW (실제: 194.33 MW) → 약 3% 오차
- Gen 2: ~45 MW (실제: 36.72 MW) → 약 22% 오차
- 총 비용: ~7500 $/h (실제: 8081.53 $/h) → 약 7% 오차
- 전압: 대략적 범위만 제시, 개별 모선 정확도 낮음

→ LLM은 "그럴듯한" 수치를 생성하지만, OPF의 정확한 해를 계산하지 못함
"""

llm_response_b = """
방법 B 결과 (CoT — 개념은 정확, 수치는 부정확):
1. 문제 정의: 발전 비용 최소화, 조류 방정식 제약 — 정확
2. 알고리즘: Newton-Raphson 기반 비선형 최적화 — 정확
3. 경향 분석: "비용이 낮은 발전기가 우선 투입" — 정확
   하지만 구체적 출력값은 여전히 추정치에 불과

→ 개념적 이해는 우수하나, 수치 계산은 근본적으로 불가
"""

llm_response_c = """
방법 C 결과 (코드 생성 — 실행 시 정확):
```python
import pandapower as pp
import pandapower.networks as pn
net = pn.case14()
pp.runopp(net, verbose=False)
print(f"총 비용: {net.res_cost:.2f} $/h")
print(net.res_gen[['p_mw', 'q_mvar']].round(2))
```
→ 생성된 코드를 실행하면 솔버와 동일한 정확한 결과를 얻음
"""

print(llm_response_a)
print(llm_response_b)
print(llm_response_c)

## 4. 결과 비교 시각화

In [ ]:
# 솔버 결과 vs LLM 추정값 비교
solver_gen = [194.33, 36.72, 28.74, 0.0, 8.49]  # 실제 OPF 결과
llm_gen = [200, 45, 30, 5, 10]  # LLM 직접 답변 (부정확)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x = np.arange(5)
width = 0.35
axes[0].bar(x - width/2, solver_gen, width, label='OPF 솔버', color='#1B7A8A')
axes[0].bar(x + width/2, llm_gen, width, label='LLM 직접 답변', color='#C75C3A')
axes[0].set_xlabel('발전기')
axes[0].set_ylabel('출력 (MW)')
axes[0].set_title('발전기 출력: 솔버 vs LLM')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'G{i+1}' for i in range(5)])
axes[0].legend()

# 오차
errors = [abs(s-l)/max(s,0.01)*100 for s, l in zip(solver_gen, llm_gen)]
axes[1].bar(x, errors, color='#D4984A')
axes[1].set_xlabel('발전기')
axes[1].set_ylabel('상대 오차 (%)')
axes[1].set_title('LLM 직접 답변의 오차')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'G{i+1}' for i in range(5)])

plt.tight_layout()
plt.show()

print("\n핵심 교훈:")
print("1. LLM은 계산기가 아니다 → 수치 최적화를 직접 시키지 말 것")
print("2. LLM은 프로그래머다 → 코드 생성에 활용할 것")
print("3. LLM은 해설자다 → 결과 해석에 활용할 것")